In [6]:
import os
import re
import time
import faiss

from bs4 import BeautifulSoup
from langchain.docstore.document import Document
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

In [7]:
def clean_text(text):
    text = re.sub(r'\n', '\n', text)
    text = re.sub(r'\t+', '\t', text)
    text = re.sub(r'\t\s+', ' ', text)
    text = re.sub(r'\n\s+', '\n', text)
    text = text.strip()
    return text

In [8]:

driver = webdriver.Chrome() 

def scrape_website_content(url):
    driver.get(url)

    WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, "main.entry-content"))
    )

    page_source = driver.page_source
    soup = BeautifulSoup(page_source, 'html.parser')
    raw_content = soup.find('main', {'class': 'entry-content'})
    content = clean_text(raw_content.get_text())
    return {"url": url, "content": content}

In [14]:
db_name="hr_info", 
base_url="http://localhost:11434"

embeddings = OllamaEmbeddings(model="nomic-embed-text", base_url=base_url)

faiss_path = os.path.join("hr_info", "index.faiss")
pkl_path = os.path.join("hr_info", "index.pkl")

if os.path.exists(faiss_path) and os.path.exists(pkl_path):
    vector_store = FAISS.load_local("hr_info", embeddings, allow_dangerous_deserialization=True)
else:
    vector_dim = len(embeddings.embed_query("Hello world"))
    index = faiss.IndexFlatL2(vector_dim)
    vector_store = FAISS(
        embedding_function=embeddings,
        index=index,
        docstore=InMemoryDocstore(),
        index_to_docstore_id={}
    )

In [18]:
content = scrape_website_content("https://manaosoftware.com/about-us/")
documents = [Document(page_content=content["content"], metadata={"url": content["url"]})]
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = text_splitter.split_documents(documents)


In [19]:
chunks

[Document(metadata={'url': 'https://manaosoftware.com/about-us/'}, page_content='A Leading Software House in Thailand \nSchedule a free consultation\nYour Trusted Software Development Partner \nOur Danish founder infused the company DNA with a Danish sense of design, quality, and craftsmanship. \nSchedule a free consultation\nOur Vision \nOur vision is to make Thailand the first choice for high-quality software development outsourcing. \nOur Mission \nWe want to be a refreshing alternative for clients struggling to outsource their software development to traditional companies. We find bright, promising local talent, develop their skills to expert levels in an empowering environment based on understanding, quality, and speed, and produce affordable and remarkably high-quality results. \nThe Story Behind Manao Software'),
 Document(metadata={'url': 'https://manaosoftware.com/about-us/'}, page_content='The Story Behind Manao Software \nDuring his university years in Denmark, Christopher M

In [ ]:
# if len(chunks) > 0:
#     print(f"[{time.strftime('%X')}] กำลังคำนวณเวกเตอร์...")
#     vector_store.add_documents(chunks)
#     vector_store.save_local(db_name)
#     print(f"[{time.strftime('%X')}] บันทึก vector store เรียบร้อยที่ '{db_name}'")
# else:
#     print(f"[{time.strftime('%X')}] ไม่พบข้อมูลในเอกสารใหม่")